# Build Dashboard
Loads all `.pkl` runs, computes metrics, and generates `dashboard.html`.

In [ ]:
import glob
import json
import numpy as np
from plotting import load_run, compute_metrics

# Load all pkl files
pkl_files = sorted(glob.glob("gd_trajectories/run_*.pkl"))
print(f"Found {len(pkl_files)} runs")

# Compute metrics and group by ratio
runs_by_ratio = {}  # ratio -> list of run dicts

for f in pkl_files:
    data = load_run(f)
    metrics = compute_metrics(data)
    if not metrics:
        continue
    
    config = data["config"]
    n, d, k, seed = config["n"], config["d"], config["k"], config["seed"]
    ratio = round(k / n, 6)
    
    run_entry = {
        "label": f"k/n = {ratio:.4f}  (k={k}, n={n}, d={d}, seed={seed})",
        "k": int(k), "n": int(n), "d": int(d), "seed": int(seed),
        "times": metrics["times"].tolist(),
    }
    
    # Add available metrics
    if "norms" in metrics:
        run_entry["norm"] = metrics["norms"].tolist()
    if "loss_values" in metrics:
        run_entry["train_loss"] = metrics["loss_values"].tolist()
        run_entry["loss_times"] = metrics["loss_times"].tolist()
    if "pop_loss_values" in metrics:
        run_entry["pop_loss"] = metrics["pop_loss_values"].tolist()
        run_entry["pop_loss_times"] = metrics["pop_loss_times"].tolist()
    if "angle_w_star" in metrics:
        run_entry["angle_w_star"] = metrics["angle_w_star"].tolist()
    if "angle_w_tilde" in metrics:
        run_entry["angle_w_tilde"] = metrics["angle_w_tilde"].tolist()
    if "stopping_times" in metrics:
        run_entry["stopping_times"] = [int(t) for t in metrics["stopping_times"]]
    if "w_star_norm" in metrics:
        run_entry["w_star_norm"] = float(metrics["w_star_norm"])
    
    if ratio not in runs_by_ratio:
        runs_by_ratio[ratio] = []
    runs_by_ratio[ratio].append(run_entry)

sorted_ratios = sorted(runs_by_ratio.keys())
print(f"Ratios: {sorted_ratios}")
print(f"Seeds per ratio: { {r: [run['seed'] for run in runs_by_ratio[r]] for r in sorted_ratios} }")

In [ ]:
# Build the JSON data blob
dashboard_data = {
    "ratios": sorted_ratios,
    "runs": {}
}

for ratio in sorted_ratios:
    key = str(ratio)
    dashboard_data["runs"][key] = runs_by_ratio[ratio]

json_blob = json.dumps(dashboard_data)
print(f"JSON size: {len(json_blob) / 1024:.1f} KB")

In [ ]:
html_template = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>GD Trajectory Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #f5f5f5; padding: 20px; }
  h1 { text-align: center; margin-bottom: 18px; font-size: 1.5em; color: #333; }
  .controls { background: #fff; border-radius: 8px; padding: 18px 24px; margin-bottom: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }
  .slider-row { display: flex; align-items: center; gap: 16px; margin-bottom: 12px; }
  .slider-row label { font-weight: 600; white-space: nowrap; }
  .slider-row input[type=range] { flex: 1; }
  .slider-row .slider-label { min-width: 340px; font-size: 0.95em; color: #555; }
  .controls-row { display: flex; align-items: center; gap: 24px; flex-wrap: wrap; }
  .controls-row label { font-weight: 600; margin-right: 4px; }
  .checkbox-group { display: flex; gap: 14px; flex-wrap: wrap; }
  .checkbox-group label { font-weight: 400; cursor: pointer; display: flex; align-items: center; gap: 4px; }
  .seed-select { padding: 4px 8px; font-size: 0.95em; }
  .plots { display: flex; gap: 16px; }
  .plots > div { flex: 1; background: #fff; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); padding: 8px; }
  @media (max-width: 900px) { .plots { flex-direction: column; } }
</style>
</head>
<body>
<h1>GD Trajectory Dashboard</h1>

<div class="controls">
  <div class="slider-row">
    <label>k/n ratio:</label>
    <input type="range" id="ratioSlider" min="0" max="0" value="0" step="1">
    <span class="slider-label" id="sliderLabel">—</span>
  </div>
  <div class="controls-row">
    <div>
      <label>Seed:</label>
      <select id="seedSelect" class="seed-select"></select>
    </div>
    <div class="checkbox-group" id="metricToggles"></div>
  </div>
</div>

<div class="plots">
  <div><div id="plotLeft" style="width:100%;height:500px;"></div></div>
  <div><div id="plotRight" style="width:100%;height:500px;"></div></div>
</div>

<script>
const DATA = __JSON_DATA__;

const METRICS = [
  { key: 'norm',        label: 'Norm',         color: '#2ca02c', timesKey: 'times' },
  { key: 'train_loss',  label: 'Train Loss',   color: '#d62728', timesKey: 'loss_times' },
  { key: 'pop_loss',    label: 'Pop Loss',     color: '#9467bd', timesKey: 'pop_loss_times' },
  { key: 'angle_w_star',label: 'Angle to w*',  color: '#1f77b4', timesKey: 'times' },
  { key: 'angle_w_tilde',label: 'Angle to w\u0303', color: '#ff7f0e', timesKey: 'times' },
];

const slider = document.getElementById('ratioSlider');
const sliderLabel = document.getElementById('sliderLabel');
const seedSelect = document.getElementById('seedSelect');
const togglesDiv = document.getElementById('metricToggles');

let prevRatioIdx = null;
let currentRatioIdx = 0;

// Build checkboxes
METRICS.forEach(m => {
  const lbl = document.createElement('label');
  const cb = document.createElement('input');
  cb.type = 'checkbox'; cb.checked = true; cb.dataset.metric = m.key;
  cb.addEventListener('change', updatePlots);
  lbl.appendChild(cb);
  lbl.appendChild(document.createTextNode(' ' + m.label));
  togglesDiv.appendChild(lbl);
});

// Init slider
slider.max = DATA.ratios.length - 1;
slider.addEventListener('input', onSliderChange);
seedSelect.addEventListener('change', updatePlots);

function getCheckedMetrics() {
  return Array.from(togglesDiv.querySelectorAll('input:checked')).map(cb => cb.dataset.metric);
}

function getRunsForRatio(ratioIdx) {
  const ratio = DATA.ratios[ratioIdx];
  return DATA.runs[String(ratio)] || [];
}

function populateSeeds(ratioIdx) {
  const runs = getRunsForRatio(ratioIdx);
  const prevSeed = seedSelect.value;
  seedSelect.innerHTML = '';
  runs.forEach((run, i) => {
    const opt = document.createElement('option');
    opt.value = i;
    opt.textContent = 'seed=' + run.seed;
    seedSelect.appendChild(opt);
  });
  // Try to keep same seed index
  const matchIdx = runs.findIndex(r => String(r.seed) === prevSeed || false);
  // Just keep index 0 if no match
  seedSelect.value = matchIdx >= 0 ? matchIdx : 0;
}

function getSelectedRun(ratioIdx) {
  const runs = getRunsForRatio(ratioIdx);
  if (runs.length === 0) return null;
  // For the current ratio, use the dropdown; for previous, try to match seed
  const idx = Math.min(parseInt(seedSelect.value) || 0, runs.length - 1);
  return runs[idx];
}

function getRun(ratioIdx) {
  const runs = getRunsForRatio(ratioIdx);
  if (runs.length === 0) return null;
  // Try to match the currently selected seed value
  const selectedRun = getSelectedRun(currentRatioIdx);
  if (selectedRun && ratioIdx !== currentRatioIdx) {
    const match = runs.find(r => r.seed === selectedRun.seed);
    if (match) return match;
  }
  const idx = Math.min(parseInt(seedSelect.value) || 0, runs.length - 1);
  return runs[idx];
}

function normalize(arr) {
  if (!arr || arr.length === 0) return [];
  const mn = Math.min(...arr);
  const mx = Math.max(...arr);
  if (mx === mn) return arr.map(() => 0.5);
  return arr.map(v => (v - mn) / (mx - mn));
}

function buildTraces(run, checkedMetrics) {
  if (!run) return [];
  const traces = [];
  METRICS.forEach(m => {
    if (!checkedMetrics.includes(m.key)) return;
    if (!run[m.key]) return;
    const raw = run[m.key];
    const times = run[m.timesKey] || run.times;
    const normed = normalize(raw);
    traces.push({
      x: times,
      y: normed,
      type: 'scatter',
      mode: 'lines',
      name: m.label,
      line: { color: m.color, width: 2 },
      text: raw.map((v, i) => `${m.label} = ${v.toFixed(4)}`),
      hovertemplate: '%{text}<br>t = %{x}<extra></extra>',
    });
  });
  // Stopping time vertical lines
  if (run.stopping_times) {
    run.stopping_times.forEach((st, i) => {
      traces.push({
        x: [st, st],
        y: [0, 1],
        type: 'scatter',
        mode: 'lines',
        line: { color: 'red', width: 1.5, dash: 'dash' },
        showlegend: i === 0,
        name: i === 0 ? 't* = ' + st : '',
        hoverinfo: 'skip',
      });
    });
  }
  return traces;
}

function plotLayout(title) {
  return {
    title: { text: title, font: { size: 14 } },
    xaxis: { title: 'Iteration t', type: 'log', gridcolor: '#eee' },
    yaxis: { title: 'Normalized value', range: [-0.05, 1.05], gridcolor: '#eee' },
    legend: { orientation: 'h', y: -0.15, x: 0.5, xanchor: 'center' },
    margin: { t: 50, b: 80, l: 50, r: 20 },
    plot_bgcolor: '#fafafa',
    hovermode: 'closest',
  };
}

function updatePlots() {
  const checked = getCheckedMetrics();

  const currentRun = getRun(currentRatioIdx);
  const currentTraces = buildTraces(currentRun, checked);
  const currentTitle = currentRun ? currentRun.label : '(no data)';
  Plotly.react('plotLeft', currentTraces, plotLayout('Current: ' + currentTitle), {responsive: true});

  if (prevRatioIdx !== null && prevRatioIdx !== currentRatioIdx) {
    const prevRun = getRun(prevRatioIdx);
    const prevTraces = buildTraces(prevRun, checked);
    const prevTitle = prevRun ? prevRun.label : '(no data)';
    Plotly.react('plotRight', prevTraces, plotLayout('Previous: ' + prevTitle), {responsive: true});
  } else {
    Plotly.react('plotRight', [], plotLayout('Previous: (none)'), {responsive: true});
  }
}

function onSliderChange() {
  const newIdx = parseInt(slider.value);
  if (newIdx !== currentRatioIdx) {
    prevRatioIdx = currentRatioIdx;
    currentRatioIdx = newIdx;
  }
  populateSeeds(currentRatioIdx);
  const run = getRun(currentRatioIdx);
  sliderLabel.textContent = run ? run.label : '—';
  updatePlots();
}

// Initial render
populateSeeds(0);
onSliderChange();
</script>
</body>
</html>"""

# Inject JSON data
html_content = html_template.replace('__JSON_DATA__', json_blob)

# Write to file
output_path = "dashboard.html"
with open(output_path, "w") as f:
    f.write(html_content)

print(f"Written {output_path} ({len(html_content) / 1024:.1f} KB)")